# Local Eval Harness v5 — Prompt Suffix + Multi-Extraction

**Hypothesis:** The comp's grader appends a `\boxed{}` suffix to every prompt (per Tong Huikang's `corpus.py`). We've been omitting it. Adding the suffix should make the model produce extractable `\boxed{}` answers more reliably.

**Suffix (verbatim from Tong's corpus.py):**
```
\nPlease put your final answer inside `\\boxed{}`. For example: `\\boxed{your answer}`
```

**Extraction:** apply 3 strategies and report each
- `after_think` — content between `</think>` and `<|im_end|>` (v4 strategy)
- `boxed_in_answer` — `\boxed{...}` content in the after-`</think>` region
- `boxed_anywhere` — `\boxed{...}` content anywhere in completion (most lenient)

**HOLDOUT_N=5** — bigger than v4's 3, smaller than v3's 10. Get more signal.

In [ ]:
# Cell 1: Config + setup (same as v4)
import os, random, shutil, site, stat, sys, time

import numpy as np
import torch

T0 = time.perf_counter()
def step(label):
    print(f"[{time.perf_counter() - T0:7.1f}s] {label}", flush=True)

SEED = 42
HOLDOUT_N = 5
MAX_NEW_TOKENS = 1024

PROMPT_SUFFIX = (
    "\nPlease put your final answer inside `\\boxed{}`. "
    "For example: `\\boxed{your answer}`"
)

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
step(f"Config: HOLDOUT_N={HOLDOUT_N}, MAX_NEW_TOKENS={MAX_NEW_TOKENS}")
step(f"Prompt suffix: {PROMPT_SUFFIX!r}")

# sitedir + ptxas
for utility_root in ("/kaggle/input/notebooks", "/kaggle/usr/lib/notebooks", "/kaggle/lib"):
    if not os.path.isdir(utility_root):
        continue
    for root, dirs, _ in os.walk(utility_root):
        if "models" in root.split(os.sep):
            dirs[:] = []
            continue
        if os.path.basename(root) == "python_packages":
            site.addsitedir(root)

PTXAS_DST = "/tmp/ptxas-blackwell"
for src in (
    "/kaggle/usr/lib/notebooks/ryanholbrook/nvidia_utility_script/triton/backends/nvidia/bin/ptxas-blackwell",
    "/kaggle/input/notebooks/ryanholbrook/nvidia-utility-script/triton/backends/nvidia/bin/ptxas-blackwell",
):
    if os.path.exists(src):
        if not os.path.exists(PTXAS_DST):
            shutil.copy(src, PTXAS_DST)
            os.chmod(PTXAS_DST, stat.S_IRWXU | stat.S_IRGRP | stat.S_IXGRP | stat.S_IROTH | stat.S_IXOTH)
        os.environ["TRITON_PTXAS_PATH"] = PTXAS_DST
        os.environ["TRITON_PTXAS_BLACKWELL_PATH"] = PTXAS_DST
        break
step("Setup done")

In [ ]:
# Cell 2: Load model + tokenizer + LoRA (same as v4)
import kagglehub
import mamba_ssm  # noqa: F401
from peft import LoraConfig, TaskType, get_peft_model
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_PATH = kagglehub.model_download("metric/nemotron-3-nano-30b-a3b-bf16/transformers/default")
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)
step("Tokenizer loaded")

IM_END_ID = tokenizer.convert_tokens_to_ids("<|im_end|>")
step(f"IM_END_ID={IM_END_ID}")

# Sanity-print formatted sample
sample_messages = [{"role": "user", "content": "What is 2+2?" + PROMPT_SUFFIX}]
sample_text = tokenizer.apply_chat_template(sample_messages, tokenize=False, add_generation_prompt=True)
step("Sample formatted prompt (with suffix):")
print(repr(sample_text))

t = time.perf_counter()
model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH, device_map="auto", trust_remote_code=True, dtype=torch.bfloat16,
)
model.eval()
step(f"Model loaded in {time.perf_counter() - t:.0f}s")

lora_cfg = LoraConfig(
    r=32, lora_alpha=16,
    target_modules=r".*\.(in_proj|out_proj|up_proj|down_proj)$",
    lora_dropout=0.05, bias="none", task_type=TaskType.CAUSAL_LM,
)
model = get_peft_model(model, lora_cfg)
step("LoRA wrapped (untrained)")

In [ ]:
# Cell 3: Multi-strategy extractors
import re

BOXED_RE = re.compile(r"\\boxed\{([^}]*)\}")
THINK_CLOSE = "</think>"
IM_END = "<|im_end|>"

def extract_after_think(completion: str) -> str:
    if THINK_CLOSE not in completion:
        return ""
    post = completion.split(THINK_CLOSE, 1)[1]
    if IM_END in post:
        post = post.split(IM_END, 1)[0]
    return post.strip()

def extract_boxed_in_answer(completion: str) -> str:
    region = extract_after_think(completion)
    m = BOXED_RE.search(region)
    return m.group(1).strip() if m else ""

def extract_boxed_anywhere(completion: str) -> str:
    matches = list(BOXED_RE.finditer(completion))
    return matches[-1].group(1).strip() if matches else ""  # last \boxed{} in text

step("Extractors ready")

In [ ]:
# Cell 4: Held-out slice
import pandas as pd
TRAIN_CSV = "/kaggle/input/competitions/nvidia-nemotron-model-reasoning-challenge/train.csv"
df = pd.read_csv(TRAIN_CSV).sort_values("id").reset_index(drop=True)
holdout = df.tail(HOLDOUT_N).reset_index(drop=True)
step(f"Held-out: {holdout.shape}; ids={list(holdout['id'])}")

In [ ]:
# Cell 5: Inference loop with PROMPT_SUFFIX appended
results = []
t_loop = time.perf_counter()

for i, row in holdout.iterrows():
    pid = row["id"]
    prompt_with_suffix = row["prompt"] + PROMPT_SUFFIX
    gold = str(row["answer"])

    messages = [{"role": "user", "content": prompt_with_suffix}]
    inputs = tokenizer.apply_chat_template(
        messages, return_tensors="pt", add_generation_prompt=True, return_dict=True,
    ).to(model.device)
    prompt_len = inputs["input_ids"].shape[1]

    t_p = time.perf_counter()
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            eos_token_id=[IM_END_ID, tokenizer.eos_token_id],
            pad_token_id=tokenizer.eos_token_id,
        )
    elapsed = time.perf_counter() - t_p
    n_new = out.shape[1] - prompt_len
    completion = tokenizer.decode(out[0, prompt_len:], skip_special_tokens=False)

    after = extract_after_think(completion)
    boxed_a = extract_boxed_in_answer(completion)
    boxed_w = extract_boxed_anywhere(completion)
    has_close = THINK_CLOSE in completion

    results.append({
        "id": pid,
        "prompt_len_tokens": int(prompt_len),
        "completion_tokens": int(n_new),
        "completion": completion,
        "after_think": after,
        "boxed_in_answer": boxed_a,
        "boxed_anywhere": boxed_w,
        "has_think_close": has_close,
        "gold": gold,
        "correct_after_think": after.strip() == gold.strip(),
        "correct_boxed_in_answer": boxed_a.strip() == gold.strip(),
        "correct_boxed_anywhere": boxed_w.strip() == gold.strip(),
        "seconds": elapsed,
    })
    print(
        f"  [{i+1}/{HOLDOUT_N}] id={pid} | {n_new:4d} tok in {elapsed:5.1f}s | close={has_close} | "
        f"after={after[:25]!r:30s} | boxed={boxed_w!r:20s} | gold={gold!r:15s}",
        flush=True,
    )

step(f"Inference loop done in {time.perf_counter() - t_loop:.0f}s")

In [ ]:
# Cell 6: Summary + save
import json
n = len(results)
summary = {
    "holdout_n": n,
    "version": "v5-suffix-multi-extract",
    "max_new_tokens": MAX_NEW_TOKENS,
    "prompt_suffix_used": True,
    "adapter": "untrained (= base model)",
    "think_closed_rate": sum(r["has_think_close"] for r in results) / max(n, 1),
    "acc_after_think": sum(r["correct_after_think"] for r in results) / max(n, 1),
    "acc_boxed_in_answer": sum(r["correct_boxed_in_answer"] for r in results) / max(n, 1),
    "acc_boxed_anywhere": sum(r["correct_boxed_anywhere"] for r in results) / max(n, 1),
    "avg_completion_tokens": sum(r["completion_tokens"] for r in results) / max(n, 1),
    "avg_seconds_per_problem": sum(r["seconds"] for r in results) / max(n, 1),
    "total_inference_seconds": sum(r["seconds"] for r in results),
}
out_df = pd.DataFrame(results)
out_df.to_csv("/kaggle/working/eval_results.csv", index=False)
with open("/kaggle/working/eval_summary.json", "w") as f:
    json.dump(summary, f, indent=2)
print("\n=== Summary ===")
print(json.dumps(summary, indent=2))

In [ ]:
# Cell 7: Inspect each completion
for i, r in enumerate(results, 1):
    print(f"=== Problem {i} (id={r['id']}, gold={r['gold']!r}) ===")
    print(f"close={r['has_think_close']} | after_think={r['after_think'][:80]!r}")
    print(f"boxed_anywhere={r['boxed_anywhere']!r}")
    print("--- COMPLETION (last 600 chars) ---")
    print(r["completion"][-600:])
    print()
step(f"Total wall time: {(time.perf_counter() - T0) / 60:.1f} min")